# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


# Before we look at the Artificial Neural Networks

## There is a different kind of Neural Network we could consider

In [4]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


Human predicted 120.0 for an item that actually costs 219.0


In [8]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

# And now - a vanilla Neural Network

During the remainder of this course we will get deeper into how Neural Networks work, and how to train a neural network.

This is just a sneak preview - let's make our own Neural Network, from scratch, using Pytorch.

Use this to get intuition; it's not important to know all about Neural networks at this point..

In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [12]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [13]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [14]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [15]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 11191.804, Val Loss: 20347.865


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 11085.901, Val Loss: 17033.641


In [16]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [17]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$45 $83 $13 $33 $62 $157 $16 $77 $48 $36 $373 $95 $122 $190 $37 $44 $16 $50 $27 $43 $19 $35 $119 $51 $256 $244 $220 $32 $90 $34 $71 $186 $10 $11 $117 $236 $53 $123 $94 $78 $155 $140 $33 $54 $97 $64 $90 $68 $41 $7 $30 $30 $21 $9 $121 $71 $47 $149 $48 $45 $36 $10 $14 $12 $399 $130 $9 $239 $16 $282 $10 $45 $92 $112 $8 $70 $114 $55 $43 $67 $51 $151 $39 $28 $20 $59 $21 $85 $141 $141 $31 $149 $35 $18 $36 $58 $64 $63 $119 $179 $33 $50 $5 $7 $9 $77 $135 $260 $11 $73 $7 $43 $164 $51 $9 $206 $189 $77 $74 $47 $25 $247 $51 $20 $39 $46 $26 $197 $85 $10 $45 $155 $135 $35 $104 $27 $95 $94 $29 $86 $61 $135 $8 $190 $254 $92 $92 $266 $36 $14 $23 $227 $14 $47 $33 $116 $168 $4 $60 $4 $83 $15 $15 $29 $444 $23 $26 $4 $41 $53 $17 $25 $206 $73 $26 $21 $19 $11 $19 $168 $350 $25 $83 $15 $63 $118 $6 $72 $39 $17 $58 $89 $110 $52 $9 $42 $114 $49 $10 $23 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [27]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [28]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [29]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [30]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [31]:
gpt_4__1_nano(test[0])

'$250'

In [32]:
test[0].price

219.0

In [33]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$39 $34 $25 $10 $20 $90 $6 $65 $11 $2170 $13 $29 $30 $19 $1 $8 $71 $0 $40 $39 $54 $26 $65 $45 $182 $274 $405 $0 $201 $64 $30 $15 $10 $60 $35 $19 $90 $26 $6 $13 $175 $45 $10 $105 $70 $5 $12 $13 $65 $52 $20 $105 $225 $0 $197 $16 $8 $30 $48 $13 $116 $2 $41 $30 $179 $30 $90 $325 $25 $74 $17 $8 $30 $6 $10 $21 $76 $5 $8 $1 $30 $0 $15 $74 $11 $10 $32 $44 $0 $21 $13 $20 $5 $10 $6 $108 $1 $93 $70 $275 $50 $33 $12 $11 $1 $32 $10 $350 $4 $49 $10 $336 $39 $78 $54 $80 $20 $5 $64 $47 $24 $211 $50 $16 $0 $10 $5 $101 $29 $59 $79 $13 $65 $5 $85 $5 $85 $10 $78 $62 $16 $0 $70 $10 $134 $118 $15 $390 $15 $13 $1 $144 $17 $7860 $1 $29 $101 $41 $30 $5 $211 $17 $7 $3 $440 $3 $752 $30 $5 $5 $5 $3 $20 $8 $22 $101 $3 $57 $56 $13 $246 $25 $250 $99 $0 $18 $73 $17 $10 $2 $5 $129 $5 $11 $50 $70 $10 $20 $21 $1 

In [37]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-sonnet-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [38]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

AuthenticationError: litellm.AuthenticationError: Missing Anthropic API Key - A call is being made to anthropic but no key is set either in the environment variables or via params. Please set `ANTHROPIC_API_KEY` in your environment vars

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [41]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [42]:
evaluate(gpt_5__1, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$10 $64 $5 $10 $10 $170 $64 $85 $11 $1 $162 $20 $3 $1 $39 $8 $21 $17 $10 $29 $6 $4 $0 $15 $57 $203 $195 $2 $81 $64 $10 $30 $10 $50 $5 $119 $30 $41 $64 $18 $160 $45 $16 $95 $40 $2 $5 $1 $75 $28 $24 $112 $245 $5 $27 $34 $6 $60 $58 $4 $106 $48 $32 $70 $129 $0 $20 $305 $5 $44 $17 $3 $100 $3 $25 $17 $26 $2 $2 $5 $0 $4 $8 $74 $14 $25 $118 $56 $0 $11 $3 $20 $10 $5 $1 $98 $1 $62 $80 $175 $20 $7 $2 $1 $50 $152 $16 $355 $4 $119 $20 $86 $9 $58 $54 $0 $8 $1 $4 $396 $8 $91 $10 $36 $10 $20 $0 $21 $1 $59 $149 $3 $5 $0 $45 $2 $25 $10 $102 $12 $6 $149 $20 $9 $24 $2 $5 $110 $15 $8 $5 $84 $27 $60 $1 $71 $41 $38 $75 $0 $130 $17 $1 $1 $59 $2 $452 $25 $0 $2 $10 $1 $161 $13 $62 $9 $6 $47 $6 $13 $105 $15 $235 $29 $25 $8 $63 $17 $10 $3 $0 $19 $7 $41 $60 $40 $0 $0 $18 $8 